# Tabular Classification Project: MLP with Embeddings (+ XGBoost Comparison)

## Purpose

This is a **self-directed project** focused on building a neural network for tabular classification using PyTorch embeddings. You'll also implement XGBoost as a baseline comparison.

**What you'll practice:**
- Data exploration and preprocessing for real-world messy data
- Handling categorical features with learned embeddings in PyTorch
- Building a full NN pipeline: DataLoaders, training loop, evaluation
- Regularization: dropout, batch normalization, weight decay
- Proper evaluation with imbalanced data (beyond accuracy)
- XGBoost as a comparison baseline

**Why neural networks on tabular data?**

Trees dominate most tabular benchmarks, but NNs have real advantages:
- Learned embeddings capture relationships between categories that label encoding misses
- Scale better with dataset size (300K+ rows)
- Can combine with other modalities (images, text) in the same model
- Transfer learning potential — pretrained embeddings can bootstrap new tasks

The goal isn't to "beat" XGBoost. It's to build the full pipeline and understand when each approach makes sense.

In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    print("Colab environment ready! All required packages are pre-installed.")
except ImportError:
    pass  # Not on Colab, no action needed

## Dataset Options

Below are tabular datasets suitable for this project. The key requirement: **mix of categorical and numerical features** so you practice building embeddings.

### Summary Table

| Dataset | Rows | Cat:Num | Task | Imbalance | Difficulty |
|---------|------|---------|------|-----------|------------|
| **Bank Marketing** | 41,188 | 10:10 | Binary | 88/12 | Medium |
| **Adult Income** | 48,842 | 8:6 | Binary | 76/24 | Medium |
| **Census-Income KDD** | 299,285 | ~24:16 | Binary | 94/6 | Hard |
| **Secondary Mushroom** | 61,069 | 17:3 | Binary | ~balanced | Medium |
| **Porto Seguro** | 595,212 | 14+17bin:26 | Binary | 96/4 | Hard |

## Dataset Deep Dive

### 1. Bank Marketing (Portuguese Bank)

**What it is:** Predict whether a client will subscribe to a term deposit after a phone marketing campaign.

**Source:** UCI ML Repository. Data from a Portuguese bank's direct marketing campaigns (2008-2013).

**Location:** `DATA_PATH / "bank_marketing" / "bank-additional" / "bank-additional-full.csv"` (semicolon-delimited)

> **Download required:** This dataset is not included in the repo. Download from UCI ML Repository. See `data/DATASETS.md` for instructions.

| Pros | Cons |
|------|------|
| Good size (41K rows) - NNs can compete | Semicolon delimiter (minor) |
| Balanced mix: 10 categorical, 10 numerical | `duration` feature leaks info (should be dropped) |
| Rich categoricals (job has 12 levels, month, day_of_week) | |
| Missing values coded as "unknown" - forces real decisions | |
| 88/12 class imbalance - teaches evaluation beyond accuracy | |
| Includes economic context features (euribor rate, employment) | |
| Interpretable domain - good for teaching material | |

**Categorical features:** job (12), marital (4), education (8), default, housing, loan, contact, month, day_of_week, poutcome
**Numerical features:** age, duration, campaign, pdays, previous, emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed

**Expected performance:** F1 ~0.5-0.6 on minority class, AUC ~0.90+

**Best for:** First project. Good categorical variety, interpretable features, real messiness without being overwhelming.

### 2. Adult Income (Census Income)

**What it is:** Predict whether a person earns >$50K/year based on census data.

**Source:** UCI ML Repository, extracted from 1994 Census database.

**Location:** `DATA_PATH / "adult_income" / "adult.data"`

> **Download required:** This dataset is not included in the repo. Download from UCI ML Repository. See `data/DATASETS.md` for instructions.

| Pros | Cons |
|------|------|
| Good size (48K rows) | Overused in tutorials |
| High-cardinality categoricals (occupation, country) | Older data (1994) |
| Missing values marked with "?" | |
| 76/24 class imbalance | |
| Well-benchmarked - easy to sanity-check results | |

**Categorical features:** workclass, education, marital-status, occupation, relationship, race, sex, native-country
**Numerical features:** age, fnlwgt, education-num, capital-gain, capital-loss, hours-per-week

**Expected performance:** Accuracy 85-87%, both approaches competitive.

**Best for:** Safe pick. You'll find reference implementations everywhere to compare against.

### 3. Census-Income KDD

**What it is:** Same domain as Adult Income but 6x the data and more features. Predict income >$50K.

**Source:** UCI ML Repository / KDD.

**Location:** `DATA_PATH / "census_income_kdd" / "census-income.data"` (no header row - column names in `.names` file)

> **Download required:** This dataset is not included in the repo. Download from UCI ML Repository. See `data/DATASETS.md` for instructions.

| Pros | Cons |
|------|------|
| Massive (300K rows) - NNs genuinely benefit from scale | No header row, need to parse column names from docs |
| ~40 features with rich categorical variety | Lots of "Not in universe" values to handle |
| 94/6 class imbalance - forces serious evaluation strategy | More preprocessing overhead |
| Pre-split train/test | Feature redundancy to manage |

**Expected performance:** Similar domain to Adult but more severe imbalance changes the challenge.

**Best for:** When you want Adult-like data at NN-competitive scale. This is where NNs start showing real advantage.

### 4. Secondary Mushroom

**What it is:** Predict whether a mushroom is edible or poisonous based on physical characteristics.

**Source:** UCI ML Repository (2021 version - much better than the classic 1987 version).

| Pros | Cons |
|------|------|
| 61K rows, solid size | Nearly balanced classes (less evaluation challenge) |
| Overwhelmingly categorical: 17 of 20 features | Anonymized somewhat |
| Interpretable (cap-shape, gill-color, habitat) | |
| Missing values present | |
| Can visualize what embeddings learn about mushroom features | |

**Categorical features:** cap-shape, cap-surface, cap-color, does-bruise-bleed, gill-attachment, gill-spacing, gill-color, stem-root, stem-surface, stem-color, veil-type, veil-color, has-ring, ring-type, spore-print-color, habitat, season
**Numerical features:** cap-diameter, stem-height, stem-width

**Expected performance:** High accuracy achievable (90%+).

**Best for:** Maximum embedding practice. 17 categoricals means you'll get very comfortable building embedding layers.

### 5. Porto Seguro (Safe Driver Prediction)

**What it is:** Predict whether a driver will file an insurance claim. From a Kaggle competition (5,200 teams).

**Source:** Kaggle competition (Porto Seguro's Safe Driver Prediction).

| Pros | Cons |
|------|------|
| Massive (595K rows) | Anonymized features (can't interpret what embeddings learn) |
| Explicitly labeled categoricals (`_cat` suffix) | Brutal 96/4 imbalance |
| Heavy missingness (coded as -1) | Larger preprocessing effort |
| Well-benchmarked from competition | Not ideal for teaching material |
| NNs placed well in competition | |

**Expected performance:** Normalized Gini ~0.29 (top Kaggle solutions).

**Best for:** Challenge project after you've got the pipeline working on something simpler.

## Recommendations

### Top Pick: Bank Marketing

**Why:**
- 10 categorical features with real variety (job has 12 levels, education has 8)
- 10 numerical features including economic indicators
- "unknown" values force real missing-data decisions
- 88/12 imbalance makes evaluation meaningful (accuracy alone is misleading)
- Interpretable domain — you can explain what embeddings learn, reusable as teaching material
- `duration` feature is a known data leak — teaches the lesson of thinking before modeling

### Second Pick: Adult Income

**Why:**
- Already downloaded locally, well-benchmarked everywhere
- High-cardinality categoricals (native-country ~41 values, occupation ~14)
- Slightly less interesting domain but tons of reference implementations to compare against

### For a Challenge: Census-Income KDD or Porto Seguro

**Why:**
- 300K-600K rows — this is where NNs genuinely outperform trees
- More severe imbalance forces you to go beyond basic metrics
- Heavier preprocessing — closer to real-world data engineering

### Skip for This Project

| Dataset | Why Skip |
|---------|----------|
| Titanic | Too small (891 rows) — NNs will overfit, XGBoost wins trivially |
| Heart Disease | Too small (303 rows) |
| Wine Quality | No categoricals — misses the entire embedding practice |
| California Housing | Regression + no categoricals |
| Secondary Mushroom | Good for pure embedding practice but nearly balanced classes reduce evaluation challenge |

## Your Choice

**Dataset I'm using for this project:**

*(Write your choice here)*

# Part A: Neural Networks (MLP with Embeddings)

MLPs can compete with XGBoost on tabular data, especially when:
- Dataset is large (>10K rows)
- High-cardinality categorical features benefit from learned embeddings
- You want to combine with other modalities later

This is the primary focus of this project. Get this working well before moving to XGBoost.

## MLP Tasks

### Phase A1: Data Exploration

**Task A1.1: Load and inspect the data**
- Load the dataset (CSV, sklearn, or other source)
- Check shape, dtypes, first few rows
- Identify target variable

**Task A1.2: Explore distributions**
- Check for missing values
- Examine target class balance
- Look at numerical feature distributions
- Count unique values in categorical columns

**Task A1.3: Basic visualizations**
- Histograms of numerical features
- Bar charts of categorical features
- Correlation heatmap (numerical only)

### Phase A2: Preprocessing

**Task A2.1: Investigate data leakage**
- The `duration` feature (call length in seconds) is a known leak in this dataset
- Think about it: can you know call duration *before* you make the call?
- Check the correlation between `duration` and the target — what do you see?
- Read the dataset authors' note about this in `bank-additional-names.txt` (line 56)
- Look into: what is "target leakage" vs "train-test contamination"? They're different failure modes
- Decide what to do with `duration` and document your reasoning

**Task A2.2: Handle missing values**
- Decide strategy: drop, impute with median/mode, or create "Unknown" category
- Implement your chosen strategy

**Task A2.3: Split data**
- Train/validation/test split (e.g., 80/10/10)
- Use stratification for imbalanced data
- Do this BEFORE encoding/scaling to avoid data leakage

**Task A2.4: Identify feature types**
- List categorical columns and numerical columns
- Store cardinality of each categorical (needed for embeddings)

**Task A2.5: Encode categorical features**
- Label encoding (integers 0 to n-1)
- Embeddings will learn representations, no one-hot needed

**Task A2.6: Scale numerical features**
- StandardScaler is essential for neural networks
- Fit on train, transform train/val/test
- Keep numerical and categorical arrays separate

### Phase A3: Data Loading (PyTorch)

**Task A3.1: Create TabularDataset class**
- Subclass `torch.utils.data.Dataset`
- Store numerical features as FloatTensor
- Store categorical features as LongTensor (for embeddings)
- Store labels
- Implement `__len__` and `__getitem__`

**Task A3.2: Create DataLoaders**
- Training loader with shuffle=True
- Validation/test loaders with shuffle=False
- Choose batch size (start with 256 or 512)

### Phase A4: Model Architecture

**Task A4.1: Define TabularNN class**
- Subclass `nn.Module`
- Create embedding layers for each categorical feature
- Embedding dimension rule: `min(50, (n_categories + 1) // 2)`

**Task A4.2: Define forward pass**
- Get embeddings for each categorical
- Concatenate: [numerical | all embedded categoricals]
- Pass through MLP layers:
  - Linear → BatchNorm → ReLU → Dropout (repeat 2-3x)
  - Final Linear for output

**Task A4.3: Verify architecture**
- Create model instance, move to GPU
- Pass dummy batch, check output shape
- Print parameter count

### Phase A5: Training

**Task A5.1: Set up training components**
- Loss: `BCEWithLogitsLoss` (binary) or `CrossEntropyLoss` (multi-class)
- Handle class imbalance with `pos_weight`
- Optimizer: Adam with `weight_decay` for L2 regularization
- Scheduler: `ReduceLROnPlateau`

**Task A5.2: Write training loop**
- For each epoch:
  - Training phase: forward, loss, backward, update
  - Validation phase: compute metrics without gradients
- Track loss and metrics history

**Task A5.3: Implement early stopping**
- Save best model based on validation metric
- Stop if no improvement for N epochs

**Task A5.4: Monitor training**
- Print progress every few epochs
- Watch for overfitting (train improves, val doesn't)

### Phase A6: Evaluation

**Task A6.1: Compute metrics**
- Load best model
- Accuracy, ROC-AUC, classification report (precision, recall, F1)
- Store predictions for later comparison

**Task A6.2: Plot learning curves**
- Training loss vs epochs
- Validation metric vs epochs

**Task A6.3: Inspect embeddings**
- Examine what the embedding layers learned
- Do similar categories cluster together?

### Bank loan dataset

This notebook takes inspiration from various places, but it's a great idea to compare what we're doing to https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html - the difference is that the docs focus on using pytorch for images, while we're working with standard tabular data here. 
And the pytorch tutorial doesn't cover all that much analysis of data preprocessing, but they do help out with some of the more formal steps like loading the dataset, creating the model, and creating a training loop. 

https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html

https://archive.ics.uci.edu/dataset/222/bank+marketing

In [ ]:
from IPython.core.interactiveshell import InteractiveShell                                                                                                                                 
InteractiveShell.ast_node_interactivity = "all"

This dataset is data on phone calls to customers which is attempting to convince them to put their money in bank term deposits (in sweden, that means you'd try to convince them to sign up for a fast-ränte-konto, e.g you get 3% if you lock your money up for 1 year)

In [ ]:
# Your imports here

from pathlib import Path

import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Download required: bank_marketing is not in the repo.
# Download from UCI ML Repository — see data/DATASETS.md for instructions.
DATA_PATH = Path("../../../../data")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

### Datasets
There are a few versions, the additional one also includes some economic indicators which seems to be relevant reading the paper. 
The entire paper is based on data from a real database, where they did feature engineering to figure out which ones that mattered based on expertise. If we'd be in the same situation, we might use a decision tree to do the same. 

  1. Duration vs target — because if it leaks, I drop it before anything else. No point analyzing a feature I won't use.                                                                     
  2. Target balance — because this decides my loss function (pos_weight), my split strategy (stratified), and which metrics matter (F1/AUC over accuracy).
  3. Categorical unique values + unknown counts — because this sets my embedding sizes and tells me if any feature is mostly missing (like default at 21%). You've done this already.        
  4. df.describe() on numericals — because I need to know what to scale and what's weird. pdays will jump out (999 as sentinel value), campaign and previous are likely skewed.              
  5. pdays specifically — 999 means "never contacted." That's not a real number, it'll mess up scaling. Need to decide: treat as categorical flag, or replace 999 with something.
  6. Correlation between numerical features — quick check if any are redundant. The economic indicators might be highly correlated with each other.

In [ ]:
# A1: Data Exploration

dataset_path = DATA_PATH / "bank_marketing" / "bank-additional" / "bank-additional-full.csv"
df = pd.read_csv(dataset_path, sep=";")

df.info()

In [ ]:
df

In [ ]:
df.select_dtypes('object').nunique()

In [ ]:
df["job"].unique()

In [ ]:
df["month"].unique()

In [ ]:
df.isin(["unknown"]).sum()

### Handling Missing Values: What Are Our Options?

This is the first real preprocessing decision in any ML pipeline. We found "unknown" values in 6 columns — now what?

There are three main strategies, each with trade-offs:

**1. Drop rows with missing values**
Remove any row that has an "unknown." Simple, but dangerous when missingness is common. Dropping all rows with `default == "unknown"` would remove 8,597 samples (21% of the data). That's a lot of signal thrown away. Only makes sense when very few rows are affected (<1%).

**2. Impute (fill in a guess)**
Replace "unknown" with the most common value (mode) for categoricals, or the median for numericals. The upside: no data lost. The downside: you're making up information. If 8,597 people refused to say whether they defaulted on a loan, filling in "no" pretends you know the answer. It also destroys the pattern — maybe people who refuse to answer behave differently than people who say "no."

**3. Keep "unknown" as its own category**
Treat "unknown" as a valid answer, not a gap to fill. This is what we're doing. The embedding layer will learn a vector for "unknown" just like it learns vectors for "married" or "admin." If refusing to answer a question correlates with subscription behavior, the model can pick that up.

**Which approach when?**

| Situation | Best strategy |
|---|---|
| Few rows affected (<1%) | Drop rows — minimal data loss |
| Numerical column, moderate missingness | Impute with median + add a `was_missing` flag column |
| Categorical column with embeddings | Keep as own category (what we're doing) |
| Column is >90% missing | Drop the entire column — not enough signal |
| Missingness itself might be informative | Keep it / add a flag — don't destroy the pattern |

For our dataset, "unknown" is already a clean category string, not a NaN. When we label encode, it just gets its own integer like any other value. No extra step needed — but the *decision* to treat it this way is the important part.

**Note:** The right strategy depends on the model. When we work with trees later, they handle missing values differently — a tree can split on "is this value missing?" directly, so filling with a sentinel value like -1 works great. With images, "missing data" means something else entirely (corrupted pixels, different image sizes). The concept stays the same — inspect, decide, handle — but the tools change each time.

In [ ]:
for col in df.select_dtypes('object').columns:                                                                                                                                             
    print(df[col].value_counts())            
    print()                                                                                                                                                                                
                

In [ ]:
df

Things that stand out:

default — this is "has credit in default?" (meaning: have they failed to pay back a loan). Only 3 people in 41K said "yes." So the feature can't really distinguish between defaulters and 
non-defaulters — there's almost no defaulters in the data. What it can distinguish is people who answered "no" vs people who answered "unknown" (8,597 people). That's really what the
embedding will learn: the difference between "definitely no default" and "refused to answer." Whether that predicts subscription is worth checking but it's a weak feature overall.        
                                                            
month — this isn't really about seasonality like "people subscribe more in December." It's about when the bank chose to run campaigns. They made 13,769 calls in May but only 182 in       
December. So if the model learns "May = low subscription rate," it might be because the bank was doing mass outreach in May (lower quality targeting, more cold calls), while December     
calls were highly targeted. The month encodes campaign strategy more than customer behavior.

day_of_week — nearly equal distribution across all five days (~8K each). This means the bank called roughly the same amount every day, and there's probably no strong pattern like "people
subscribe more on Fridays." When a feature has near-uniform distribution and likely no relationship with the target, it won't contribute much. The network won't be hurt by including it,
but don't expect it to be an important feature.

We might drop day_of_week and default for these reasons, basically: they dont add much, most likely. But month might be valuable since it encodes WHEN the bank was calling.  

## Data leakage?

In [ ]:
df["duration"]

### Data leakage -> The duration column

Imagine you're the bank. You want a model that answers:                                                                                                                              
                                               
  "Should we call this client? Will they subscribe?"                                                                                                                                         
                                                                  
  You want to answer this before you pick up the phone. That's the whole point — decide who to call.                                                                                         
                                                                                                                                                                                             
  Now look at your features. Most of them are things you know before calling:
  - Age, job, education — you have these from the client database
  - Economic indicators — public data you can look up
  - Previous campaign results — historical records

  But duration? You only know how long the call lasted after the call is over. And by then, you already know if they subscribed.

  So if you train your model with duration included, it'll get great accuracy — but it's cheating. It's like predicting whether it rained by looking at whether the ground is wet.
  Technically correct, but you can't use "is the ground wet?" to decide whether to bring an umbrella before you go outside.

  That's data leakage: a feature that contains information from the future — from after the thing you're trying to predict has already happened.

  Drop it because any model trained with it will look good in your notebook but would be useless in production.

Why is it even included?
It's there so researchers can compare model performance in two settings:                                                                                                                   
- With duration — how well can you predict given all information, including post-call data? (benchmark/upper bound)                                                                        
- Without duration — how well can you predict using only information available before the call? (realistic scenario)

The gap between those two results tells you how much of the prediction depends on what happens during the call vs what you knew beforehand. That's actually interesting from a research
perspective — it tells the bank how much of the outcome is driven by client profile vs call quality.

In [ ]:
df.groupby('y')['duration'].mean()

In [ ]:
df['duration'].corr(df['y'].map({'no': 0, 'yes': 1}))

In [ ]:
df = df.drop(columns=["duration"]) # We drop it due to data leakage. Again, we want a model that can predict based on factors we dont know before.

### Target balance

In [ ]:
df["y"].value_counts()

Yepp that's a huge class imbalance. We'll have to consider it later when setting up the loss

In [ ]:
df.describe()

### pdays
pdays is the numer of days since they were contacted

most are 999 (sentinel value for not contacted before)

So since so few actual have a proper value for days, we could just care a boolean flag for it instead, and drop the pdays column. 



In [ ]:
(df['pdays'] == 999).sum()                                                                                                                                                                 

In [ ]:
df['was_contacted'] = (df['pdays'] != 999).astype(int)                                                                                                                                     
df = df.drop(columns=['pdays'])   

In [ ]:
# A2: Preprocessing

## Preprocessing: Preparing Data for a PyTorch MLP

Neural networks only understand numbers. Our dataset has strings ("admin.", "married", "yes") and numbers at wildly different scales (age: 17-98, nr.employed: 4963-5228). We need to convert everything into a format the network can work with.

### The two paths through the network

Our features split into two types that get processed differently:

**Categorical features** (strings like "admin.", "married") → convert to integers → feed into **embedding layers**
- The integer is just a lookup key, not a meaningful number
- The embedding layer learns a vector representation for each category

**Numerical features** (numbers like age, euribor3m) → scale to mean=0, std=1 → feed directly into **linear layers**
- Scaling matters because the network multiplies inputs by weights. If one feature is 5000 and another is 0.6, the large one dominates learning

### The preprocessing steps (in order)

| Step | What | Why |
|------|------|-----|
| 1. Encode target | `y`: "yes"/"no" → 1/0 | Fixed mapping, safe to do before split |
| 2. Train/val split | 80/20, stratified on `y` | Must happen before steps 3-4 to prevent leakage |
| 3. Label encode categoricals | "admin." → 0, "blue-collar" → 1, ... | Fit on train only, transform both splits |
| 4. Scale numericals | StandardScaler (mean=0, std=1) | Fit on train only, transform both splits |

### Why "fit on train, transform both"?

The encoder/scaler *learns* from data (what categories exist, what the mean/std are). If it learns from validation data too, your model indirectly sees information it shouldn't have. Same principle as the duration leak, just subtler.

In [ ]:
df['y'] = df['y'].map({'yes': 1, 'no': 0}) # turn yes and no into 1 and 0                                                                                                                                           

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split

# train_test_split(stratify=True)

### Two ways of splitting

The first one includes the target (the label column) in both dataframes

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['y']) 

This one splits the labels into its own variables.
This one usually is more common as when we're working with pytorch well use tensors, and the tensors need them to be split up.
The stratify parameter ensures that both the training data and the validation data has the same ratio of yes and no labels. In our example, the entire dataset has a 88/12 ratio between yes and no. 

In [ ]:
df["y"].value_counts(normalize=True) # shows the ratio

In [ ]:
X = df.drop(columns=['y'])                                                                                                                                                                 
y = df['y']                                  
                                                                                                                                                                                            
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
cat_columns = X_train.select_dtypes("object").columns.tolist()
cat_columns

In [ ]:
num_columns = X_train.select_dtypes("number").columns.tolist()
num_columns

### Label Encoding Categoricals

Label encoding converts each category to an integer:

| Category | Integer |
|---|---|
| admin. | 0 |
| blue-collar | 1 |
| entrepreneur | 2 |
| housemaid | 3 |
| ... | ... |

The specific numbers don't matter — they're just IDs. The network doesn't think "entrepreneur (2) is between blue-collar (1) and housemaid (3)." The numbers are just lookup keys.

**Why lookup keys?** These integers feed into embedding layers. When the network sees `2` for job, it looks up row 2 in the job embedding table and gets a learned vector. The integer is just an address, not a value.

**Why fit on training data only?** `LabelEncoder` builds a mapping of category → integer. If you fit on all data, it might see categories in validation that appear in a different order or frequency. In practice this matters less than with scaling, but it's good habit.

**How it works:**

```python
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])  # fit + transform on train
    X_val[col] = le.transform(X_val[col])           # only transform on val
    label_encoders[col] = le  # store for decoding later
```

### Do we need embeddings for all categoricals?

An alternative is one-hot encoding (`pd.get_dummies`), which creates a binary column per category. But for embedding-based MLPs, we need integer IDs as lookup keys — one-hot would defeat the purpose.

Embeddings shine when there are many categories, because the learned vectors can capture similarity (e.g. "retired" and "student" might behave similarly). For 2-3 categories, one-hot carries the same information.

| Column | Unique | Embedding worth it? |
|---|---|---|
| job | 12 | Yes — can learn relationships between job types |
| education | 8 | Yes — has ordered structure (basic.4y → university.degree) |
| month | 10 | Yes — can learn which months had similar campaign patterns |
| day_of_week | 5 | Borderline — 5 values, one-hot would work too |
| marital | 4 | Borderline |
| poutcome | 3 | One-hot would be fine |
| default | 3 | One-hot would be fine |
| housing | 3 | One-hot would be fine |
| loan | 3 | One-hot would be fine |
| contact | 2 | Binary — a single 0/1 column would suffice |

**Practical decision:** embed everything. One consistent pipeline is simpler than mixing one-hot and embeddings. The network won't be hurt by embedding a 2-category feature — it just learns two vectors. Keeps the code clean.

In [ ]:
label_encoders = {}                                                                                                                                                                        
for col in cat_columns:                                                                                                                                                                    
    le = LabelEncoder()                                                                                                                                                                    
    X_train[col] = le.fit_transform(X_train[col])                                                                                                                                          
    X_val[col] = le.transform(X_val[col])
    label_encoders[col] = le

fit_transform does two things: learn the mapping, then apply it.                                                                                                                           
                                                                                                                                                                                        
transform only applies an already-learned mapping.  

We save the label reference to the encodings here:

In [ ]:
# label_encoders["default"].classes_
label_encoders["housing"].classes_

In [ ]:
label_encoders["marital"].classes_

### And now they've turned into numbers in the dataframe
This is what encoding does - it turns categorical data (text) into numbers

In [ ]:
X_train[cat_columns].head()

### Scaling Numerical Features

Neural networks learn by multiplying inputs by weights and adjusting those weights based on how wrong the prediction was. Here's the problem: if one feature ranges from 0-5228 (nr.employed) and another from 0-7 (previous), the network's gradients will be dominated by the large feature. It'll spend most of its learning capacity adjusting weights for nr.employed while barely touching previous.

StandardScaler fixes this by centering each feature at 0 and squishing it to standard deviation 1. After scaling, all features live in roughly the same range (-2 to +2 for most values), so no single feature hogs the gradient.

**Before scaling:** age = 40, nr.employed = 5191, euribor3m = 4.8
**After scaling:** age = 0.0, nr.employed = 0.3, euribor3m = 0.7

The formula is simple: `(value - mean) / std` for each column.

**Why trees don't need this:** Trees split on "is age > 38?" — the actual number matters, not its scale. Neural networks multiply inputs by weights — scale matters a lot.

**Why fit on train only:** The scaler learns the mean and std from whichever data you give it. If you include validation data, those statistics leak information about the validation set into your training pipeline. Same principle as label encoding — learn from train, apply to both.

In [ ]:
scaler = StandardScaler()

scaler.fit_transform(X_train[num_columns])

As you can see, we're scaling our numerical columns, meaning they're a lot smaller now compared to before.

In [ ]:
X_train[num_columns]

In [ ]:
scaler = StandardScaler()
# Lets overwrite it
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_val[num_columns] = scaler.transform(X_val[num_columns])

In [ ]:
X_train

### Data Loading (Dataset, DataLoader)

Everything so far lives in pandas DataFrames. PyTorch doesn't speak pandas — it speaks tensors. This step is the bridge.

**Dataset = storage.** It knows *what* the data is. You give it your DataFrame, it converts everything to tensors and stores them. Then it answers two questions: "how many samples?" (`__len__`) and "give me sample #N" (`__getitem__`). That's the entire job. It doesn't know anything about training.

**DataLoader = logistics.** It knows *how to serve* the data. It picks 256 random indices, calls `dataset[i]` for each one, stacks the results into batched tensors, and hands them to your training loop. It doesn't know or care what the data looks like.

Why separate them? Because the concerns are independent. Switch from tabular data to images — your Dataset changes completely, but the DataLoader stays identical. Change batch size from 256 to 512 — the DataLoader changes, but the Dataset stays the same.

**Why a class?** The DataLoader needs to call `__len__` and `__getitem__` on whatever you give it. In Python, you define those by writing a class. By inheriting from `Dataset` and implementing those two methods, your class behaves like a list that DataLoader can iterate over. The variable names and how you store data internally are entirely up to you.

**Why separate tensors?** Categoricals and numericals take different paths through the model. Categoricals go through embedding lookups (require integers → `LongTensor`), numericals go straight into linear layers (require floats → `FloatTensor`). You can't do an embedding lookup on a float, so one big tensor wouldn't work.

**What batching looks like:** The DataLoader stacks individual samples into batch tensors. 256 individual `(cats, nums, label)` tuples become tensors of shape `(256, 10)`, `(256, 9)`, and `(256,)`. Your training loop just unpacks them: `for cats, nums, labels in train_loader:`

Ultimately the __getitem__ method is supposed to return one row with the appropriate columns, in our case since we using embeddings we need to do some additional work, and also the label value for that particular row. 
Think of it like this:
The Dataset class, in our class our own TabularDataset, has the responsibility of being able to return particular rows from our entire dataset. 
It should return them as tensors. In the future, we'll work with other types of neural nets, we might have a dataset class that is able to return images instead. It differs depending on the data we have, which is why we create our own classes for it.

In [ ]:
from torch.utils.data import  Dataset, DataLoader

class TabularDataset(Dataset):                                                                                                                                                             
    def __init__(self, X, y, cat_columns, num_columns):                                                                                                                                    
        # Convert categorical columns to a LongTensor                                                                                                                                      
        # Convert numerical columns to a FloatTensor                            
        # Convert labels to a FloatTensor
        self.cats = torch.tensor(X[cat_columns].values, dtype=torch.long)                                                                                                                          
        self.nums = torch.tensor(X[num_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(y.values, dtype=torch.float32) 


    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.cats[idx], self.nums[idx], self.labels[idx]
    
    
train_dataset = TabularDataset(X_train, y_train, cat_columns, num_columns)
val_dataset = TabularDataset(X_val, y_val, cat_columns, num_columns)

In [ ]:
train_dataset[0]

### Dataloader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)                                                                                                                     
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False) 

In [ ]:
train_cats, train_nums, train_labels = next(iter(train_loader))

In [ ]:
len(train_cats)
len(train_nums)

train_cats # 256 rows with our categorical columns
train_nums # 256 rows our numerical columns
train_labels[1] # the label vales (1 for yes, 0 for no) for 256 of the rows

In [ ]:
# A4: Model Architecture

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(20, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

embedding_sizes = [(len(label_encoders[col].classes_), min(50, (len(label_encoders[col].classes_) + 1) // 2))
                    for col in cat_columns]
embedding_sizes

In [ ]:
class NeuralNetwork(nn.Module):                                                                                                                                                            
    def __init__(self, embedding_sizes, num_numerical_features):                                                                                                                                            
        super().__init__()                                                                                                                                                                 
                                                                                
        # Create one embedding table per categorical column
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, embedding_dim)
            for num_categories, embedding_dim in embedding_sizes
        ])

        # Total input width = sum of all embedding dims (29) + numerical features (9) = 38
        total_embedding_dim = sum(embedding_dim for _, embedding_dim in embedding_sizes)
        input_size = total_embedding_dim + num_numerical_features

        # The MLP layers
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )
        
    def forward(self, categorical_features, numerical_features):
        # Look up each categorical column in its embedding table
        embedded = [emb(categorical_features[:, i]) for i, emb in enumerate(self.embeddings)]

        # Concatenate all embedding vectors + numerical features into one wide vector
        combined = torch.cat(embedded + [numerical_features], dim=1)

        # Through the MLP
        return self.layers(combined)

embedding_sizes = [(len(label_encoders[col].classes_), min(50, (len(label_encoders[col].classes_) + 1) // 2))
                    for col in cat_columns]

model = NeuralNetwork(embedding_sizes, len(num_columns))
model

### The Forward Pass Summarized

**For one row**, the data flows through the MLP like this:

38 input vector per row in our dataset → multiplied with a weight table of 512 neurons → input becomes a vector of 512 values → multiplied with a table of weights for 512 neurons → output layer gets a vector of 512 values that is multiplied with a weight matrix of one neuron which results in one logit.

Or more concisely:

```
38 values → Linear(512) → ReLU → Linear(512) → ReLU → Linear(1) → logit
```

Each linear layer multiplies the input by a weight matrix and adds a bias. ReLU zeroes out negatives between layers, giving the network the ability to learn non-linear patterns. The final layer outputs a single number: the logit.

**From logit to loss:** We use `BCEWithLogitsLoss`, which applies sigmoid to the logit (turning it into a probability between 0 and 1) and then computes how far that probability is from the true label. That loss drives backpropagation.

**In practice, we process batches.** Our forward pass doesn't handle one row at a time — it processes 256 rows simultaneously:

```
256 rows of categoricals  →  embedding lookup  →  256 rows of 29 values
256 rows of numericals     →  passed through    →  256 rows of 9 values
                                                    ────────────────────
                              concatenate        →  256 rows of 38 values
                              through MLP        →  256 logits
```

### The Embedding Tables

Each categorical column gets its own lookup table of learnable numbers. For example, `nn.Embedding(12, 6)` for the job column creates a table with 12 rows (one per job type) and 6 columns (the embedding dimension):

| | dim0 | dim1 | dim2 | dim3 | dim4 | dim5 |
|---|---|---|---|---|---|---|
| 0 (admin.) | 0.23 | -0.41 | 0.12 | 0.87 | -0.33 | 0.05 |
| 1 (blue-collar) | 0.67 | 0.11 | -0.54 | 0.29 | 0.88 | -0.12 |
| 2 (entrepreneur) | -0.15 | 0.43 | 0.71 | -0.62 | 0.14 | 0.37 |
| ... | ... | ... | ... | ... | ... | ... |
| 11 (student) | -0.44 | 0.66 | 0.19 | -0.31 | 0.55 | -0.08 |

When the model sees job = 5, it goes to row 5 and pulls out that row's 6 numbers. Those numbers start random and get learned during training — just like weights in a linear layer.

We create 10 of these tables, one per categorical column:

| Column | Table size | Why |
|---|---|---|
| job | 12 x 6 | 12 job types, each described by 6 numbers |
| marital | 4 x 2 | 4 statuses, 2 numbers each |
| education | 8 x 4 | 8 levels, 4 numbers each |
| default | 3 x 2 | 3 values (no/unknown/yes) |
| housing | 3 x 2 | same |
| loan | 3 x 2 | same |
| contact | 2 x 1 | 2 types, just 1 number each |
| month | 10 x 5 | 10 months, 5 numbers each |
| day_of_week | 5 x 3 | 5 days, 3 numbers each |
| poutcome | 3 x 2 | 3 outcomes, 2 numbers each |

Total embedding output: 6+2+4+2+2+2+1+5+3+2 = **29 numbers**. Add 9 numerical features = **38 inputs** to the first linear layer.

In [ ]:
# A5-A6: Training & Evaluation

## Training

### 1. Lets define a loss function that works with binary classification
The function that actually tells us how wrong our prediction is once we've performed our forward pass
- CrossEntrypyLoss would be great if we had a multi classification problem, but thats not the case now. We're doing BINARY classification (2 options --> yes or no)
- nn.BCEWithLogitsLoss() is meant for binary classification, so we'll use that 

In [ ]:
# loss_fn = nn.CrossEntropyLoss() # Nope, this is if we'd be doing MULTI CLASS classification (e.g is it a cat, dog or monkey?)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()  

### However, we're in this weird situation where we have CLASS IMBALANCE. This is not always case, meaning we have a ton of data that results in "no" and just a few predictions that results in "yes" in our dataset which we're training on.

So, we have to do something about that, meaning we'll need to add "imbalance weighting"

In [ ]:
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]) # the imbalance weighting
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

### 2. An optimizer - e.g SGD or Adam optimizer - the thing that updates the weights.

After backpropagation computes the gradients (which direction each weight should move), the optimizer's job is simple: **actually update the weights**. It reads the gradients and adjusts 
every weight in the model to reduce the loss.                                                                                                                                            

The simplest optimizer (SGD) does: `weight = weight - learning_rate * gradient` for every parameter. Adam builds on this by tracking each weight's gradient history — weights that have
been getting big, consistent gradients get smaller steps (already moving fast, don't overshoot), while weights with small or noisy gradients get bigger steps (need more push). The result:
Adam effectively adapts the learning rate per weight, so it converges faster and is less sensitive to your initial learning rate choice.

From the PyTorch docs:
"Optimization is the process of adjusting model parameters to reduce model error in each training step. Optimization algorithms define how this process is performed. All optimization
logic is encapsulated in the optimizer object."

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # learning rate of 0.001 (the 3 in 1e-3 means you want 3 zeroes before the number)

## The training loop
It's now time to actually train our model.
Remember, we've had to do a lot of work to get here.
- We analyzed the dataset and figured out how to prep our data
- We did data preprocessing, like scaling numerics, encoding categorical values, splitting the data
- We created a dataset class and a dataloader
- We created the model with the layers it should have using a custom class that implements a few methods like forward
- We picked a loss function suitable for our task (binary classification, weighted because we got class imbalance in this case)
- We created an optimizer - adam is pretty standard, SGD would also work but would be worse most likely.


Now finally, we're going to train the model. Lets put it all together.

In [ ]:
class NeuralNetwork(nn.Module):                                                                                                                                                            
    def __init__(self, embedding_sizes, num_numerical_features):                                                                                                                                            
        super().__init__()                                                                                                                                                                 
                                                                                
        # Create one embedding table per categorical column
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, embedding_dim)
            for num_categories, embedding_dim in embedding_sizes
        ])

        # Total input width = sum of all embedding dims (29) + numerical features (9) = 38
        total_embedding_dim = sum(embedding_dim for _, embedding_dim in embedding_sizes)
        input_size = total_embedding_dim + num_numerical_features

        # The MLP layers
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )
        
    def forward(self, categorical_features, numerical_features):
        # Look up each categorical column in its embedding table
        embedded = [emb(categorical_features[:, i]) for i, emb in enumerate(self.embeddings)]

        # Concatenate all embedding vectors + numerical features into one wide vector
        combined = torch.cat(embedded + [numerical_features], dim=1)

        # Through the MLP
        return self.layers(combined)

embedding_sizes = [(len(label_encoders[col].classes_), min(50, (len(label_encoders[col].classes_) + 1) // 2))
                    for col in cat_columns]

# Actual model
model = NeuralNetwork(embedding_sizes, len(num_columns))

# Loss function
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]) # the imbalance weighting
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # learning rate of 0.001 (the 3 in 1e-3 means you want 3 zeroes before the number)

### A starting point

In [ ]:
# for epoch in range(num_epochs):
#     model.train()
#     for cats, nums, labels in train_loader:
#         # forward, loss, backward, step

#     model.eval()
#     with torch.no_grad():
#         for cats, nums, labels in val_loader:
#             # forward, loss (just tracking)

### A first attempt

In [ ]:

# num_epochs = 10

# for epoch in range(num_epochs):
#     model.train()
#     for cats, nums, labels in train_loader:
#         # forward, loss, backward, step
#         logit = model.forward(cats, nums)
#         loss = loss_fn(logit)
#         model.step()

#     model.eval()
#     with torch.no_grad():
#         for cats, nums, labels in val_loader:
#             # forward, loss (just tracking)

### Corrected attempt
Compare it to: https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html#full-implementation

In [ ]:
# num_epochs = 10

# for epoch in range(num_epochs):
#     model.train()
#     for cats, nums, labels in train_loader: # get the different values for each input column
#         # forward, loss, backward, step 
#         logit = model(cats, nums) #fixed
        
        
#         # The .squeeze() removes the extra dimension (model outputs shape [256, 1], e.g from processing a batch or 256 rows we get 256 logits:  [256, 1] →  [[0.34], [-1.2], [0.78], ...] -> after .squeeze(): →  [0.34, -1.2, 0.78, ...]
#         # In simpler terms: Our loss functon wants to compare our list of labels (the correct predictions) to the list of losses produced from running predictions (forward pass) during our training.  
#         loss = loss_fn(logit.squeeze(), labels) 
        
#         optimizer.zero_grad()  # clear old gradients
#         loss.backward()        # compute new gradients
#         optimizer.step()       # update weights

#     # Now lets see how the model performs on our validation set -> we'll track accuracy
#     # Read the text cell below
#     model.eval()                                                                                                                                                                           
#     val_loss_total = 0
#     with torch.no_grad():                                                                                                                                                                  
#         for cats, nums, labels in val_loader:                                                                                                                                            
#             logit = model(cats, nums)
#             val_loss_total += loss_fn(logit.squeeze(), labels).item()

#     avg_val_loss = val_loss_total / len(val_loader)

Add printstatements to track loss

Remember - this wont tell us how good the model is, but it will tell us something -> if its overfitting. 

In [ ]:
# num_epochs = 10                                                                                                                                                                            

# for epoch in range(num_epochs):                                                                                                                                                            
#     model.train()                                                                                                                                                                        
#     train_loss_total = 0
#     for cats, nums, labels in train_loader:
#         logit = model(cats, nums)
#         loss = loss_fn(logit.squeeze(), labels)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         train_loss_total += loss.item()

#     model.eval()
#     val_loss_total = 0
#     with torch.no_grad():
#         for cats, nums, labels in val_loader:
#             logit = model(cats, nums)
#             val_loss_total += loss_fn(logit.squeeze(), labels).item()

#     avg_train_loss = train_loss_total / len(train_loader)
#     avg_val_loss = val_loss_total / len(val_loader)
#     print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

After each epoch of training, we run the model on data it hasn't trained on — the validation set. Same forward pass (make predictions, compute loss), but no weight updates. We're just
asking: "how well does the model perform on unseen data?"
                                                                                                                                                                                            
This is how you catch overfitting. If training loss keeps dropping but validation loss starts climbing, the model is memorizing the training data instead of learning general patterns.
It's like a student who memorizes answers to practice exams but can't solve new problems.                                                                                                  
                                                                                                                                                                                          
model.eval() switches the model to evaluation mode — this disables things like dropout that should only be active during training. torch.no_grad() tells PyTorch "don't track gradients"
since we're not updating weights here, which saves memory and speeds things up.

### Lets add some print statements so we more easily can track the progress of the training loop. E.g we want to see how the training loss and the validation loss is progressing for each epoch. 
- Remember, one epoch means we've run forward pass on every single row in the training dataset. 
- During one epoch, we'll have the training dataset size / batch size number of batches. So e.g if we had a training dataset of 35000 and batch size of 256, we'll have 137 batches
- For each batch, we calculate a loss, that is the average of the loss of the forward pass for all those 256 rows from the dataset. Meaning, we do ONE forward pass for each batch, but that forward pass processes 256 rows at once. So we technically get 256 losses, which are averaged into one loss.

In [ ]:
InteractiveShell.ast_node_interactivity = "last_expr"

## Hyperparameter Tuning

Our training loop works, but the model's behavior depends heavily on choices we made: how many neurons, what learning rate, whether we use dropout. These are **hyperparameters** — settings you choose before training that the model can't learn on its own.

The question we're trying to answer: **how do we find settings where the model learns effectively without memorizing?**

We'll run three experiments, each telling a different story:
1. **No regularization** — what happens when you give the model too much freedom?
2. **Heavy regularization** — what happens when you constrain it too much?
3. **Balanced** — can we find the sweet spot?

Along the way, we'll introduce **dropout** and **weight decay** as tools to control overfitting, and we'll move to GPU for faster iteration.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

### Attempt 1: Our original settings (512 neurons, lr=1e-3, no regularization)

Same model we built earlier, just with GPU and loss tracking added. This is our baseline — let's see what happens.

In [ ]:
# Original model — no dropout, no weight decay, 512 neurons
model_v1 = NeuralNetwork(embedding_sizes, len(num_columns)).to(device)
optimizer_v1 = torch.optim.Adam(model_v1.parameters(), lr=1e-3)
loss_fn_v1 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

num_epochs = 20

for epoch in range(num_epochs):
    model_v1.train();
    train_loss_total = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logit = model_v1(cats, nums)
        loss = loss_fn_v1(logit.squeeze(), labels)
        optimizer_v1.zero_grad()
        loss.backward()
        optimizer_v1.step()
        train_loss_total += loss.item()

    model_v1.eval();
    val_loss_total = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logit = model_v1(cats, nums)
            val_loss_total += loss_fn_v1(logit.squeeze(), labels).item()

    avg_train_loss = train_loss_total / len(train_loader)
    avg_val_loss = val_loss_total / len(val_loader)
    print(f"Epoch {epoch+1:2d}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

Val loss improved for the first ~5 epochs (0.96 → 0.94) but then started climbing fast. By epoch 20 train loss had dropped to 0.63 while val loss ballooned to 1.37. The model learned something useful early on, but quickly started **memorizing** the training data instead of learning general patterns.

The 512-neuron network with no regularization has too much capacity for 33K rows — it can afford to memorize every training example rather than extract patterns. This is called **overfitting**.

### Regularization: preventing the model from memorizing

Think of it like studying for an exam. A student who memorizes every practice question word-for-word will ace the practice set but fail on new questions. A student who's forced to study under constraints — less time, no notes, random practice sets — has to actually *understand* the material. That understanding transfers to new questions.

Regularization is those constraints. We deliberately make training harder so the model has to learn general patterns instead of shortcuts. Here are the main tools:

**Dropout** randomly turns off a percentage of neurons during each training step. If 30% of neurons are disabled (`Dropout(0.3)`), the network can't rely on any specific neuron or small group of neurons to carry the prediction. It's forced to spread knowledge across many neurons, making the whole network more robust. During validation and prediction, dropout is turned off — all neurons contribute.

**Weight decay** adds a small penalty for having large weights. Without it, the model can assign huge importance to specific features or patterns it found in training data. Weight decay gently pushes all weights toward zero, keeping the model from becoming too "opinionated." You add it directly to the optimizer: `Adam(model.parameters(), weight_decay=1e-4)`.

**Fewer neurons** is the simplest form of regularization: give the model less room to work with. A 128-neuron layer can store less information than a 512-neuron layer, so it physically can't memorize as much. The trade-off: it might also not have enough capacity to learn the real patterns.

**Lower learning rate** doesn't prevent overfitting directly, but it slows the model down. With smaller weight updates each step, the model takes a more careful path through the loss landscape, which often leads to more generalizable solutions rather than sharp, overfit minima.

### Attempt 2: Heavy regularization (128 neurons, dropout=0.3, lr=3e-4)

Let's go the opposite direction — aggressive regularization. Smaller network, high dropout, lower learning rate, weight decay added.

In [ ]:
# Heavy regularization — 128 neurons, dropout=0.3, weight decay
class NeuralNetworkV2(nn.Module):
    def __init__(self, embedding_sizes, num_numerical_features):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(nc, ed) for nc, ed in embedding_sizes
        ])
        total_emb = sum(ed for _, ed in embedding_sizes)
        input_size = total_emb + num_numerical_features
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )
    def forward(self, cat, num):
        emb = [e(cat[:, i]) for i, e in enumerate(self.embeddings)]
        return self.layers(torch.cat(emb + [num], dim=1))

model_v2 = NeuralNetworkV2(embedding_sizes, len(num_columns)).to(device)
optimizer_v2 = torch.optim.Adam(model_v2.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn_v2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for epoch in range(30):
    model_v2.train();
    train_loss_total = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logit = model_v2(cats, nums)
        loss = loss_fn_v2(logit.squeeze(), labels)
        optimizer_v2.zero_grad()
        loss.backward()
        optimizer_v2.step()
        train_loss_total += loss.item()

    model_v2.eval();
    val_loss_total = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logit = model_v2(cats, nums)
            val_loss_total += loss_fn_v2(logit.squeeze(), labels).item()

    avg_train_loss = train_loss_total / len(train_loader)
    avg_val_loss = val_loss_total / len(val_loader)
    print(f"Epoch {epoch+1:2d}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

Both train and val loss drop steadily over 30 epochs — no overfitting at all. The heavy regularization (high dropout, small network, weight decay) is keeping the model in check. But notice the val loss is still slowly improving at epoch 30 (0.94 → 0.94) — the model is learning, just very slowly. It might be **underfitting**: held back from reaching its full potential by all the regularization.

The sweet spot is somewhere between attempt 1 (too free, memorizes) and attempt 2 (too constrained, learns slowly).

### Attempt 3: Balanced settings (256 neurons, dropout=0.15, lr=5e-4)

A middle ground: more capacity than attempt 2, lighter regularization, slightly faster learning rate. The goal is to see clear improvement *and* eventual overfitting — the classic training curve.

In [ ]:
# Balanced — 256 neurons, light dropout, slightly faster lr
class NeuralNetworkV3(nn.Module):
    def __init__(self, embedding_sizes, num_numerical_features):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(nc, ed) for nc, ed in embedding_sizes
        ])
        total_emb = sum(ed for _, ed in embedding_sizes)
        input_size = total_emb + num_numerical_features
        self.layers = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 1),
        )
    def forward(self, cat, num):
        emb = [e(cat[:, i]) for i, e in enumerate(self.embeddings)]
        return self.layers(torch.cat(emb + [num], dim=1))

model_v3 = NeuralNetworkV3(embedding_sizes, len(num_columns)).to(device)
optimizer_v3 = torch.optim.Adam(model_v3.parameters(), lr=5e-4, weight_decay=1e-5)
loss_fn_v3 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for epoch in range(40):
    model_v3.train();
    train_loss_total = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logit = model_v3(cats, nums)
        loss = loss_fn_v3(logit.squeeze(), labels)
        optimizer_v3.zero_grad()
        loss.backward()
        optimizer_v3.step()
        train_loss_total += loss.item()

    model_v3.eval();
    val_loss_total = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logit = model_v3(cats, nums)
            val_loss_total += loss_fn_v3(logit.squeeze(), labels).item()

    avg_train_loss = train_loss_total / len(train_loader)
    avg_val_loss = val_loss_total / len(val_loader)
    print(f"Epoch {epoch+1:2d}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

Val loss improves for ~6 epochs (0.96 → 0.94), then gradually climbs while train loss keeps dropping. By epoch 40, train is at 0.80 and val is at 1.02 — the gap is widening but much more slowly than attempt 1. The light regularization bought us more useful training time before overfitting kicked in.

This is the **classic training curve**: genuine improvement, then gradual overfitting. This is exactly what **early stopping** is for — save the model at its best point (epoch ~6) and stop before overfitting gets worse.

| Attempt | Neurons | Dropout | LR | Best val loss | Behavior |
|---|---|---|---|---|---|
| 1 | 512 | 0 | 1e-3 | ~0.94 (epoch 5) | Fast overfitting — too much capacity, no regularization |
| 2 | 128 | 0.3 | 3e-4 | ~0.94 (epoch 28) | Slow steady improvement — possibly underfitting |
| 3 | 256 | 0.15 | 5e-4 | ~0.94 (epoch 6) | Improves then overfits — classic training curve |

All three reach roughly the same best val loss (~0.94), but through very different paths. The key difference isn't *what* they achieve, but *how they get there* and *what happens after*. Attempt 3 gives us the most informative training curve, and with early stopping, we can capture its best moment.

### Speeding up experimentation

We manually wrote the training loop three times — that's fine for learning, but tedious for exploring. Let's wrap it in a function so we can quickly test different configs and compare results.

In [ ]:
def train_model(hidden_size, dropout_rate, lr, weight_decay, epochs=40, patience=None, verbose=True):
    """Train a model with given hyperparameters. Returns (model, best_val_loss, best_epoch)."""
    
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.embeddings = nn.ModuleList([
                nn.Embedding(nc, ed) for nc, ed in embedding_sizes
            ])
            total_emb = sum(ed for _, ed in embedding_sizes)
            input_size = total_emb + len(num_columns)
            self.layers = nn.Sequential(
                nn.Linear(input_size, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_size, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_size, 1),
            )
        def forward(self, cat, num):
            emb = [e(cat[:, i]) for i, e in enumerate(self.embeddings)]
            return self.layers(torch.cat(emb + [num], dim=1))

    model = Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    best_val_loss = float('inf')
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train();
        train_loss_total = 0
        for cats, nums, labels in train_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logit = model(cats, nums)
            loss = loss_fn(logit.squeeze(), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item()

        model.eval();
        val_loss_total = 0
        with torch.no_grad():
            for cats, nums, labels in val_loader:
                cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
                logit = model(cats, nums)
                val_loss_total += loss_fn(logit.squeeze(), labels).item()

        avg_train = train_loss_total / len(train_loader)
        avg_val = val_loss_total / len(val_loader)
        
        if verbose:
            print(f"Epoch {epoch+1:2d}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_epoch = epoch + 1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        elif patience:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1} — best val_loss was {best_val_loss:.4f}")
                break

    if best_state:
        model.load_state_dict(best_state)
    
    return model, best_val_loss, best_epoch

Now we can quickly sweep over different values for each hyperparameter while keeping the others fixed. This is called a **grid search** — systematic experimentation.

In [ ]:
# How does dropout rate affect performance? (256 neurons, lr=5e-4 fixed)
print("=== Dropout comparison (256 neurons, lr=5e-4) ===\n")
for dropout_rate in [0.0, 0.1, 0.2, 0.3, 0.5]:
    _, best_val, best_epoch = train_model(
        hidden_size=256, dropout_rate=dropout_rate, lr=5e-4, weight_decay=1e-5,
        epochs=25, patience=None, verbose=False
    )
    print(f"dropout={dropout_rate}  →  best val_loss={best_val:.4f} (epoch {best_epoch})")

In [ ]:
# How does network size affect performance? (dropout=0.15, lr=5e-4 fixed)
print("=== Network size comparison (dropout=0.15, lr=5e-4) ===\n")
for hidden_size in [64, 128, 256, 512]:
    _, best_val, best_epoch = train_model(
        hidden_size=hidden_size, dropout_rate=0.15, lr=5e-4, weight_decay=1e-5,
        epochs=25, patience=None, verbose=False
    )
    print(f"hidden={hidden_size:3d}  →  best val_loss={best_val:.4f} (epoch {best_epoch})")

In [ ]:
# How does learning rate affect performance? (256 neurons, dropout=0.15 fixed)
print("=== Learning rate comparison (256 neurons, dropout=0.15) ===\n")
for lr in [1e-2, 1e-3, 5e-4, 1e-4, 5e-5]:
    _, best_val, best_epoch = train_model(
        hidden_size=256, dropout_rate=0.15, lr=lr, weight_decay=1e-5,
        epochs=25, patience=None, verbose=False
    )
    print(f"lr={lr:<8}  →  best val_loss={best_val:.4f} (epoch {best_epoch})")

### What did we learn?

All configs land in a surprisingly tight range (~0.93-0.95 best val loss). The hyperparameters change the training *dynamics* — how fast the model converges, when it starts overfitting — but the model's ceiling on this dataset is similar regardless.

A few patterns worth noting:
- **Dropout**: no dropout overfits fastest (best at epoch 5), higher dropout takes more epochs to peak. The sweet spot is around 0.1-0.3.
- **Network size**: smaller networks (64-128) take longer to converge but don't overfit as easily. Larger networks converge faster but need more regularization. For 33K rows, even 64 neurons is enough.
- **Learning rate**: too high (1e-2) can still work but is unstable. Too low (5e-5) learns too slowly within 25 epochs. The range 1e-4 to 1e-3 is the safe zone for Adam.

This is common with tabular data — the model quickly hits a ceiling determined more by the data than the architecture. Unlike images or text where bigger models keep improving, tabular datasets have limited signal to extract.

### Training our final model

Let's pick our best config and train it properly with early stopping, so we can evaluate it in the next section.

In [ ]:
model, best_val_loss, best_epoch = train_model(
    hidden_size=128,
    dropout_rate=0.15,
    lr=3e-4,
    weight_decay=1e-5,
    epochs=50,
    patience=7,
    verbose=True
)
print(f"\nBest val_loss: {best_val_loss:.4f} at epoch {best_epoch}")

## Evaluation

The training loop gave us loss numbers, but loss doesn't tell us how good the model actually is. Now we need to measure performance in terms that are meaningful: how many predictions were right, where did the model get confused, and how does it compare to just guessing?

We'll split this into two parts:
- **Basic evaluation** — the standard metrics every classifier needs: accuracy, confusion matrix, and getting predictions from the model
- **Evaluation for imbalanced data** — our dataset has 88% "no" and 12% "yes", which makes some basic metrics misleading. We'll look at precision, recall, F1, and AUC-ROC to get the full picture

### Basic Evaluation

#### Getting predictions from the model

The model outputs logits — raw numbers that can be anything from -10 to +10. To turn these into actual yes/no predictions:
1. Apply **sigmoid** to squish them into probabilities (0 to 1)
2. **Threshold** at 0.5: above = "yes", below = "no"

In [ ]:
from sklearn.metrics import accuracy_score

# Get predictions on the validation set
model.eval();
all_logits = []
all_labels = []

with torch.no_grad():
    for cats, nums, labels in val_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logits = model(cats, nums).squeeze()
        all_logits.append(logits)
        all_labels.append(labels)

all_logits = torch.cat(all_logits)     # raw model output
all_labels = torch.cat(all_labels)     # true labels

# Sigmoid turns logits into probabilities (0 to 1)
probabilities = torch.sigmoid(all_logits)

# Threshold at 0.5: above = predict "yes" (1), below = predict "no" (0)
predictions = (probabilities >= 0.5).float()

accuracy = accuracy_score(all_labels.cpu(), predictions.cpu())
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")

Accuracy tells us "what percentage of all predictions were correct." Simple and intuitive, but it treats every mistake equally — missing a subscriber is the same as wrongly flagging a non-subscriber. For some problems that's fine. For ours, we'll see why it isn't in the advanced section.

#### Confusion Matrix

A confusion matrix breaks down the predictions into four categories instead of one number:

|  | Predicted No | Predicted Yes |
|---|---|---|
| **Actually No** | True Negative (correct) | False Positive (wrong alarm) |
| **Actually Yes** | False Negative (missed) | True Positive (caught it) |

This shows us *where* the model is right and wrong, not just *how often*.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels.cpu(), predictions.cpu())
disp = ConfusionMatrixDisplay(cm, display_labels=["No", "Yes"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# Also print the raw numbers
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn} (correctly predicted 'no')")
print(f"False Positives: {fp} (predicted 'yes' but was actually 'no')")
print(f"False Negatives: {fn} (predicted 'no' but was actually 'yes')")
print(f"True Positives:  {tp} (correctly predicted 'yes')")
print(f"\nOut of {tp + fn} actual subscribers, the model caught {tp} ({tp/(tp+fn)*100:.1f}%)")

### Evaluation for Imbalanced Data

Our dataset is 88% "no" and 12% "yes." This means a model that always predicts "no" gets 88.7% accuracy — without learning anything at all. Our model's ~83% accuracy is actually *lower* than that, which looks terrible. But it's not.

We used `pos_weight=7.88` to tell the model "catching subscribers matters more than avoiding false alarms." The model responded by predicting "yes" more aggressively, which **lowers accuracy** (more false positives) but **catches more actual subscribers** (fewer false negatives). Whether that trade-off is worth it depends on the business — is it cheaper to call someone who won't subscribe, or to miss someone who would?

To see this trade-off clearly, we need metrics designed for imbalanced data:

- **Precision**: when the model says "yes", how often is it right? High precision = few wasted calls.
- **Recall**: of all actual subscribers, how many did the model find? High recall = few missed opportunities.
- **F1 score**: balances precision and recall into one number. This is the go-to metric for imbalanced classification.
- **AUC-ROC**: how well the model ranks subscribers above non-subscribers, regardless of threshold.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_labels.cpu(), predictions.cpu(), target_names=["No", "Yes"]))

### AUC-ROC

AUC-ROC answers: "if I pick a random subscriber and a random non-subscriber, how likely is the model to give the subscriber a higher probability?" A score of 0.5 = random guessing, 1.0 = perfect separation. Unlike accuracy and F1, AUC uses the raw probabilities rather than the 0.5 threshold, so it measures overall ranking quality.

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

auc = roc_auc_score(all_labels.cpu(), probabilities.cpu())
print(f"AUC-ROC: {auc:.4f}")

RocCurveDisplay.from_predictions(all_labels.cpu(), probabilities.cpu())
plt.title(f"ROC Curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.legend()
plt.show()

## Reference Materials

**PyTorch:**
- [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)
- [torch.nn.BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [torch.nn.BatchNorm1d](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html)
- [torch.optim.lr_scheduler.ReduceLROnPlateau](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html)

## Reflection

After completing the project, answer these questions:

1. Which hyperparameter settings worked best? Why do you think that is?

2. Did the MLP overfit? What signs did you see, and what helped prevent it?

3. What role did dropout and weight decay play in your best model?

4. How did pos_weight affect training? What happened without it?

5. If you had to deploy this model, what additional steps would you take?

6. What would you do differently if the dataset had 500K rows instead of 41K?

*Updated: 2026_02_16_1400*

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>